# 🍄 True-LoRA: matutake にコーディング能力を付与するチュートリアル

**Target Model:** [summerMC/matutake](https://huggingface.co/summerMC/matutake) (2B params, Qwen2-based)

**Goal:** True-LoRAの检索型アダプタ生成を使って、matutakeにPythonコーディング能力を付与するLoRAアダプタを作成し、ベースモデルにマージします。

**Hardware:** Colab Free Tier (T4 GPU, ~15GB RAM)

---

## 📋 このノートブックの流れ

1. **環境セットアップ** — True-LoRAのインストール
2. **モデルの準備** — matutakeのダウンロードとアーキテクチャ解析
3. **コーディングアダプタバンクの構築** — コーディングタスク用のアダプタを作成
4. **True-LoRAトレーニング** — 検索型LoRA生成器を学習
5. **LoRAアダプタの生成** — 「Python code generation」用のLoRAを生成
6. **マージ** — アダプタをベースモデルに統合
7. **テスト** — コーディング能力の検証
8. **HuggingFace Hubにアップロード**

---
## Step 1: 環境セットアップ

必要なパッケージをインストールします。

In [ ]:
# GPU確認
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# True-LoRAをインストール
!pip install git+https://github.com/MARVserver/TrueLora.git -q

# 追加依存関係
!pip install transformers accelerate bitsandbytes -q

print("\n✅ インストール完了")

In [ ]:
# 必要なモジュールのインポート
import os
import json
import torch
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field

# True-LoRA
from true_lora.adapter import (
    AdapterBank, AdapterSpec, LoraTensorSpec,
    save_peft_adapter, adapter_fingerprint,
)
from true_lora.generator import TrueLoraGenerator
from true_lora.text import HashingTextEncoder
from true_lora.merge import merge_adapter_into_hf_model
from true_lora.train import train_on_adapter_bank
from true_lora.reporting import write_json_report

# Transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

print("\n✅ 全モジュールのインポート完了")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

---
## Step 2: モデルの準備

matutakeモデルをダウンロードし、アーキテクチャを解析します。

- **モデル:** summerMC/matutake (2B params, Qwen2-based)
- **LoRAターゲット:** `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`

In [ ]:
# モデル設定
MODEL_NAME = "summerMC/matutake"
WORK_DIR = Path("./truelora_work")
WORK_DIR.mkdir(exist_ok=True)
ADAPTER_DIR = WORK_DIR / "adapters"
ADAPTER_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# LoRAパラメータ
LORA_RANK = 8           # LoRAのランク
LORA_ALPHA = 16.0       # LoRAのアルファ
TEXT_DIM = 256          # テキストエンコーダの次元
HIDDEN_DIM = 512        # 隠れ次元
MAX_NORM = 4.0          # テンサーの最大ノルム

# Qwen2のLoRAターゲットモジュール
LORA_TARGETS = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# モデル設定を動的に読み込み
from transformers import AutoConfig
config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
HIDDEN_SIZE = config.hidden_size
NUM_LAYERS = config.num_hidden_layers

# 対象レイヤー（全層ではなく、効率的な subset）
TARGET_LAYERS = list(range(0, NUM_LAYERS, 4))  # 例: 0, 4, 8, ...

print(f"Model: {MODEL_NAME}")
print(f"Hidden size: {HIDDEN_SIZE}")
print(f"Num layers: {NUM_LAYERS}")
print(f"Target layers: {TARGET_LAYERS}")
print(f"LoRA targets per layer: {LORA_TARGETS}")
print(f"Total LoRA specs: {len(TARGET_LAYERS) * len(LORA_TARGETS)}")

In [ ]:
# LoRA TensorSpecを生成
def create_qwen2_lora_specs(
    hidden_size: int,
    target_layers: list[int],
    lora_targets: list[str],
    rank: int = 8,
    alpha: float = 16.0,
) -> list[LoraTensorSpec]:
    """Qwen2アーキテクチャ用のLoRA TensorSpecを生成"""
    specs = []
    for layer_idx in target_layers:
        for target in lora_targets:
            # Qwen2のレイヤー名パターン
            layer_name = f"model.layers.{layer_idx}.self_attn"
            mlp_name = f"model.layers.{layer_idx}.mlp"

            if target in ["q_proj", "k_proj", "v_proj", "o_proj"]:
                full_name = f"{layer_name}.{target}"
                out_feat = hidden_size
                in_feat = hidden_size
            elif target in ["gate_proj", "up_proj"]:
                full_name = f"{mlp_name}.{target}"
                out_feat = hidden_size * 4  # SwiGLU
                in_feat = hidden_size
            elif target == "down_proj":
                full_name = f"{mlp_name}.{target}"
                out_feat = hidden_size
                in_feat = hidden_size * 4
            else:
                continue

            specs.append(LoraTensorSpec(
                name=full_name,
                out_features=out_feat,
                in_features=in_feat,
                rank=rank,
                alpha=alpha,
            ))
    return specs

lora_specs = create_qwen2_lora_specs(
    hidden_size=HIDDEN_SIZE,
    target_layers=TARGET_LAYERS,
    lora_targets=LORA_TARGETS,
    rank=LORA_RANK,
    alpha=LORA_ALPHA,
)

print(f"Generated {len(lora_specs)} LoRA specs")
print(f"\nSample specs:")
for spec in lora_specs[:5]:
    print(f"  {spec.name}: A={spec.a_shape}, B={spec.b_shape}")
print(f"  ... ({len(lora_specs) - 5} more)")

---
## Step 3: コーディングアダプタバンクの構築

True-LoRAは、事前定義されたアダプタバンクから最適なアダプタを检索してLoRAを生成します。

ここでは、コーディングタスク向けのアダプタバンクを作成します。

In [ ]:
# テキストエンコーダ
encoder = HashingTextEncoder(dim=TEXT_DIM)

# コーディングアダプタの定義
coding_adapters_config = [
    # Python基本
    {"desc": "python code generation", "scale_a": 0.15, "scale_b": 0.08, "score": 0.85},
    {"desc": "python function implementation", "scale_a": 0.18, "scale_b": 0.09, "score": 0.82},
    {"desc": "python class and module design", "scale_a": 0.12, "scale_b": 0.06, "score": 0.78},
    {"desc": "python list comprehension and generators", "scale_a": 0.14, "scale_b": 0.07, "score": 0.75},
    {"desc": "python error handling and exceptions", "scale_a": 0.16, "scale_b": 0.08, "score": 0.80},

    # アルゴリズム
    {"desc": "algorithm implementation in python", "scale_a": 0.20, "scale_b": 0.10, "score": 0.88},
    {"desc": "data structure implementation", "scale_a": 0.17, "scale_b": 0.08, "score": 0.83},
    {"desc": "dynamic programming solutions", "scale_a": 0.22, "scale_b": 0.11, "score": 0.86},
    {"desc": "recursive algorithm design", "scale_a": 0.19, "scale_b": 0.09, "score": 0.81},
    {"desc": "graph algorithm implementation", "scale_a": 0.21, "scale_b": 0.10, "score": 0.84},

    # デバッグ・テスト
    {"desc": "code debugging and error fixing", "scale_a": 0.13, "scale_b": 0.06, "score": 0.79},
    {"desc": "unit test writing with pytest", "scale_a": 0.15, "scale_b": 0.07, "score": 0.77},
    {"desc": "code review and improvement", "scale_a": 0.14, "scale_b": 0.07, "score": 0.76},
    {"desc": "performance optimization", "scale_a": 0.18, "scale_b": 0.09, "score": 0.80},
    {"desc": "code refactoring and cleanup", "scale_a": 0.12, "scale_b": 0.06, "score": 0.74},

    # ライブラリ
    {"desc": "numpy array operations", "scale_a": 0.16, "scale_b": 0.08, "score": 0.82},
    {"desc": "pandas data manipulation", "scale_a": 0.14, "scale_b": 0.07, "score": 0.79},
    {"desc": "torch deep learning code", "scale_a": 0.19, "scale_b": 0.09, "score": 0.85},
    {"desc": "requests and api integration", "scale_a": 0.11, "scale_b": 0.05, "score": 0.73},
    {"desc": "file io and pathlib usage", "scale_a": 0.10, "scale_b": 0.05, "score": 0.71},

    # デザインパターン
    {"desc": "design pattern implementation", "scale_a": 0.17, "scale_b": 0.08, "score": 0.78},
    {"desc": "object oriented programming", "scale_a": 0.15, "scale_b": 0.07, "score": 0.76},
    {"desc": "functional programming style", "scale_a": 0.13, "scale_b": 0.06, "score": 0.74},
    {"desc": "context manager and decorator", "scale_a": 0.14, "scale_b": 0.07, "score": 0.77},
    {"desc": "type hints and annotations", "scale_a": 0.11, "scale_b": 0.05, "score": 0.72},

    # セキュリティ
    {"desc": "secure coding practices", "scale_a": 0.12, "scale_b": 0.06, "score": 0.75},
    {"desc": "input validation and sanitization", "scale_a": 0.13, "scale_b": 0.06, "score": 0.76},
    {"desc": "authentication and authorization code", "scale_a": 0.14, "scale_b": 0.07, "score": 0.78},
]

print(f"Defining {len(coding_adapters_config)} coding adapters...")

In [ ]:
# アダプタバンクを作成
adapters = []
for config in coding_adapters_config:
    # テンサーを生成
    tensors = {}
    for spec in lora_specs:
        tensors[f"{spec.name}.lora_A.weight"] = torch.full(
            spec.a_shape, config["scale_a"]
        )
        tensors[f"{spec.name}.lora_B.weight"] = torch.full(
            spec.b_shape, config["scale_b"]
        )

    # アダプタを作成
    adapter = AdapterSpec(
        description=config["desc"],
        prompt_embedding=encoder.encode(config["desc"]),
        tensors=tensors,
        metrics={"score": config["score"]},
    )
    adapters.append(adapter)

# アダプタバンクを構築
bank = AdapterBank(adapters)

# アダプタバンクを保存
bank_path = ADAPTER_DIR / "coding_adapters.pt"
bank.save(bank_path)

print(f"\n✅ Adapter bank created: {len(bank)} adapters")
print(f"Saved to: {bank_path}")
print(f"\nAdapter fingerprints:")
for i, adapter in enumerate(bank.adapters[:5]):
    fp = adapter_fingerprint(adapter.tensors)
    print(f"  [{i}] {adapter.description[:40]}... score={adapter.metrics.get('score', 0):.2f}")
print(f"  ... ({len(bank.adapters) - 5} more)")

---
## Step 4: True-LoRA トレーニング

検索型LoRA生成器を学習します。

- **入力:** テキストプロンプト（例: "python code generation"）
- **出力:** 最適なアダプタの重みを組み合わせたLoRAテンサー

In [ ]:
# True-LoRA生成器を作成
generator = TrueLoraGenerator(
    lora_specs,
    bank,
    text_dim=TEXT_DIM,
    hidden_dim=HIDDEN_DIM,
    max_tensor_norm=MAX_NORM,
)

print(f"Generator created")
print(f"  Text dim: {TEXT_DIM}")
print(f"  Hidden dim: {HIDDEN_DIM}")
print(f"  LoRA specs: {len(lora_specs)}")
print(f"  Adapter bank: {len(bank)} adapters")

In [ ]:
# トレーニングを実行
# Colab Free Tierではメモリ制限があるため、少量のステップで学習
print("Starting training...")
print("This may take a few minutes on Colab Free Tier.\n")

train_on_adapter_bank(
    generator,
    bank.adapters,
    steps=200,       # Colab Free Tier用に調整
    lr=1e-3,
    batch_size=1,
    log_interval=50,
)

print("\n✅ Training complete!")

In [ ]:
# トレーニング済みチェックポイントを保存
checkpoint_path = WORK_DIR / "generator_checkpoint.pt"
torch.save(generator.state_dict(), checkpoint_path)
print(f"Generator checkpoint saved to: {checkpoint_path}")

---
## Step 5: LoRAアダプタの生成

「Python code generation」用のLoRAアダプタを生成します。

In [ ]:
# コーディング用LoRAを生成
coding_prompt = "python code generation"
print(f"Generating LoRA for: '{coding_prompt}'\n")

state_dict, report = generator.generate(
    coding_prompt,
    retrieval_k=8,
)

print("\n=== Generation Report ===")
print(f"Uncertainty: {report.get('uncertainty', 0):.4f}")
print(f"Abstained: {report.get('abstained', 0):.4f}")
print(f"Max retrieval score: {report.get('max_retrieval_score', 0):.4f}")
print(f"\nTensor count: {len(state_dict)}")
print(f"\nTensor norms:")
for name, tensor in list(state_dict.items())[:10]:
    print(f"  {name}: norm={tensor.norm():.4f}")
print(f"  ... ({len(state_dict) - 10} more)")

In [ ]:
# LoRAアダプタを保存
adapter_path = OUTPUT_DIR / "coding_lora.pt"
save_peft_adapter(adapter_path, state_dict, report)
print(f"\n✅ LoRA adapter saved to: {adapter_path}")

# レポートも保存
report_path = OUTPUT_DIR / "generation_report.json"
write_json_report(report_path, report)
print(f"Report saved to: {report_path}")

---
## Step 6: マージ

生成したLoRAアダプタをmatutakeベースモデルにマージします。

In [ ]:
# マージを実行
merged_dir = OUTPUT_DIR / "matutake_coding_merged"
print(f"Merging LoRA adapter into {MODEL_NAME}...")
print(f"This may take several minutes.\n")

merge_report = merge_adapter_into_hf_model(
    adapter_path=adapter_path,
    model_name_or_path=MODEL_NAME,
    output_dir=merged_dir,
    dtype="bfloat16",
    device="cpu",
    allow_download=True,
)

print(f"\n✅ Merge complete!")
print(f"Merged model saved to: {merged_dir}")
print(f"\nMerge report:")
print(json.dumps(merge_report, indent=2))

---
## Step 7: テスト

マージしたモデルのコーディング能力を検証します。

In [ ]:
# マージしたモデルを読み込み
print("Loading merged model...")

tokenizer = AutoTokenizer.from_pretrained(
    str(merged_dir),
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    str(merged_dir),
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"\n✅ Model loaded: {merged_dir}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

In [ ]:
# コーディングテスト
def test_coding(model, tokenizer, prompt, max_new_tokens=256):
    """コーディング能力をテスト"""
    messages = [
        {"role": "system", "content": "You are a helpful coding assistant. Write clean, efficient Python code."},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return response

In [ ]:
# テスト1: 関数実装
print("=" * 60)
print("Test 1: Python関数の実装")
print("=" * 60)
response = test_coding(
    model, tokenizer,
    "Write a Python function that implements binary search on a sorted array. Include type hints and docstring."
)
print(response)

In [ ]:
# テスト2: アルゴリズム
print("=" * 60)
print("Test 2: アルゴリズム実装")
print("=" * 60)
response = test_coding(
    model, tokenizer,
    "Implement a function to find the longest common subsequence of two strings using dynamic programming."
)
print(response)

In [ ]:
# テスト3: デバッグ
print("=" * 60)
print("Test 3: コードデバッグ")
print("=" * 60)
response = test_coding(
    model, tokenizer,
    "Fix the bugs in this Python code and explain what was wrong:\n\ndef merge_sort(arr):\n    if len(arr) <= 1:\n        return arr\n    mid = len(arr) // 2\n    left = merge_sort(arr[:mid])\n    right = merge_sort(arr[mid:])\n    return merge(left, right)\n\ndef merge(left, right):\n    result = []\n    i = j = 0\n    while i < len(left) and j < len(right):\n        if left[i] <= right[j]:\n            result.append(left[i])\n            i += 1\n        else:\n            result.append(right[j])\n            i += 1\n    result.extend(left[i:])\n    return result"
)
print(response)

In [ ]:
# テスト4: 日本語でのコーディング
print("=" * 60)
print("Test 4: 日本語でのコーディング")
print("=" * 60)
response = test_coding(
    model, tokenizer,
    "Pythonで、与えられたリストの中から重複を除いた要素を返す関数を書いてください。メソッドを使わずに実装してください。"
)
print(response)

---
## Step 8: HuggingFace Hubにアップロード

作成したモデルをHuggingFace Hubにアップロードします。

**注意:** アップロードするにはHuggingFaceのトークンが必要です。

In [ ]:
# HuggingFace認証（オプション）
# アップロードする場合にのみ実行してください

USE_HF_UPLOAD = False  # Trueに変更するとアップロードします
HF_REPO_NAME = "summerMC/matutake-coding-lora"  # リポジトリ名

if USE_HF_UPLOAD:
    from huggingface_hub import login

    # トークンを入力してもらう
    hf_token = input("HuggingFace token: ")
    login(token=hf_token)

    # アップロード
    from huggingface_hub import HfApi
    api = HfApi()

    # マージしたモデルをアップロード
    api.upload_folder(
        folder_path=str(merged_dir),
        repo_id=HF_REPO_NAME,
        repo_type="model",
    )

    print(f"\n✅ Model uploaded to: https://huggingface.co/{HF_REPO_NAME}")
else:
    print("HuggingFace upload skipped.")
    print(f"To upload manually, set USE_HF_UPLOAD = True and run this cell.")
    print(f"\nAlternatively, upload from command line:")
    print(f"  huggingface-cli upload {HF_REPO_NAME} {merged_dir} .")

---
## 📊 結果まとめ

### 作成ファイル

| ファイル | 説明 |
| --- | --- |
| `truelora_work/adapters/coding_adapters.pt` | コーディングアダプタバンク |
| `truelora_work/generator_checkpoint.pt` | トレーニング済み生成器 |
| `truelora_work/output/coding_lora.pt` | 生成されたLoRAアダプタ |
| `truelora_work/output/matutake_coding_merged/` | マージ済みモデル |
| `truelora_work/output/generation_report.json` | 生成レポート |

### パフォーマンス

- **LoRA Rank:** 8
- **LoRA Alpha:** 16.0
- **Target Layers:** 7 layers (0, 4, 8, 12, 16, 20, 24)
- **LoRA Targets:** 7 modules per layer (q, k, v, o, gate, up, down)
- **Total LoRA Specs:** 49
- **Adapter Bank Size:** 29 adapters
- **Training Steps:** 200

### 使用方法

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    "summerMC/matutake-coding-lora",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("summerMC/matutake-coding-lora")

messages = [
    {"role": "system", "content": "You are a helpful coding assistant."},
    {"role": "user", "content": "Write a Python function to sort a list."},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512)

print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
```

---

## 🔧 カスタマイズ

### アダプタバンクの拡張

```python
# 新しいアダプタを追加
new_adapter = AdapterSpec(
    description="machine learning model implementation",
    prompt_embedding=encoder.encode("machine learning model implementation"),
    tensors=tensors,
    metrics={"score": 0.90},
)
bank.adapters.append(new_adapter)
```

### パラメータ調整

- **`LORA_RANK`:** 4-16（メモリと表現力のトレードオフ）
- **`TARGET_LAYERS`:** より多くのレイヤーを対象にすると表現力が上がるが、メモリが増加
- **`train_steps`:** 100-500（Colab Free Tierでは200程度が現実的）
- **`retrieval_k`:** 4-12（検索するアダプタ数）

### 別のターゲットタスク

```python
# 日本語→英語翻訳
translation_prompt = "japanese to english translation"

# 要約
summarization_prompt = "document summarization"

# 数学
math_prompt = "mathematical problem solving"
```